#Preprocessing: Warm Season LS and OT Frequency

## Purpose: Prepares the warm-season Lightning Strike (LS) and Overshooting Top (OT) frequency datasets used in Figure 1.

Inputs:
- Per-year raw LS files
- Per-year raw OT files

Processing steps:
1. Load and merge all years of LS and OT data
2. Regrid OT data to BARRA-R2 grid to match LS grid
3. Convert both datasets to boolean (presence/absence per grid cell per time step), 
   using a threshold of 0.5 for OT and 1 for LS.
4. Sum booleans across time to get seasonal frequency counts only for warm season [October to April]
5. Processing is parallelised with Dask due to the size of the merged multi-year dataset.

Outputs:
- `warm_season_LS.nc` — summed LS frequency, warm season, all years
- `warm_season_OT.nc` — summed OT frequency, warm season, all years, regridded

Used by: `02_fig01_spatial_distribution_LS_OT.ipynb`

In [ ]:
%%time
import os
# import tarfile
import glob
import xarray as xr
import pandas as pd
from datetime import datetime
import xesmf as xe
import intake
import numpy as np
#import proplot
import dask

from dask.distributed import Client 
dask.distributed.Client(threads_per_worker=1)

In [6]:
def gridding(ds_coarse, ds_out,lat_max,lat_min, lon_min, lon_max):

    """
    Regrid ds_coarse onto the grid of ds_out (subset to matching bounds).
    Uses nearest neighbour (source-to-destination).
    """

    # 1. Get spatial bounds from coarse dataset
    
    # lat_min = round(ds_coarse.lat.min().item(), 4)
    # lat_max = round(ds_coarse.lat.max().item(), 4)
    # lon_min = round(ds_coarse.lon.min().item(), 4)
    # lon_max = round(ds_coarse.lon.max().item(), 4)

    # 2. Subset output grid to same region
    ds_out = ds_out.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )
    # 3. Ensure latitude ordering matches (descending)
    ds_out = ds_out.sortby('lat',ascending=False)

    # 4. Build regridder (nearest neighbour)
    gridder = xe.Regridder(ds_coarse, ds_out, 
                           method='nearest_s2d',) # method = conservative is poorest form

    # 5. Apply regridding
    ds_gridded = gridder(ds_coarse)
    
    return ds_gridded

def _preprocess(ds):

    """
    preprocessing the data before analysing
    """
    # 1. Drop unwanted variables
    ds = ds.drop_vars([v for v in drop_variables_1 if v in ds])

    # 2. Subset to region of interest (note lat slice is reversed)
    ds = ds.sel(
        grid_lat=slice(lat_max, lat_min),
        grid_lon=slice(lon_min, lon_max)
    )

    # 3. Rename coordinates to standard names
    ds = ds.rename({
        "grid_lat": "lat",
        "grid_lon": "lon"
    })

    # 4. Convert to boolean mask (thresholding OT probability)
    ds = (ds['ot_probability'] >= 0.5)

    # 5. Coarsen grid (3x3) and take maximum within each block
    ds = ds.coarsen(
        lat=3,
        lon=3,
        boundary='trim',
        coord_func="first"
    ).max()

    return ds

def _preprocess_2(ds):

    """
    this is preprocess for data starting from mid may 2023 as they have different variable name
    """
    
    # ds = ds.drop_vars(drop_variables_2)
    ds = ds.drop_vars([v for v in drop_variables_1 if v in ds])
    ds = ds.sel(grid_lat = slice(lat_max,lat_min), grid_lon = slice(lon_min,lon_max)) 
    ds = ds.rename({"grid_lat" : "lat", "grid_lon" : "lon"})
    ds = ds.rename({"maximum_ot_probability":"ot_probability"}) 
    ds = (ds['ot_probability'] >= 0.5)
    ds = ds.coarsen(lat = 3, lon = 3, boundary='trim',coord_func="first").max()

    return ds


def process_folders( folders,
                     data_path,
                     preprocess_fn,
                     # land_select = land_select, #fraction of land
                     chunks = {'lat':-1, 'lon':-1},
                     exclude_files = None,    # list of specific files to skip
                     skip_empty_files = False,  # skip 0-byte files
                     skip_errors = False,       # skip corrupt/missing batches
                   ):

    """
    Process as set of year folders -applies masking, gridding
    
    Args:
        folders        : list of folder names (Bedka's two part)
        data_path      : base path to NetCDF files
        preprocess_fn  : preprocessing function for xr.open_mfdataset
        chunks         : dask chunking strategy base don data
        exclude_files  : list of full file paths to exclude
        skip_empty_files : whether to skip empty files
        skip_errors    : whether to skip bad batches or raise
        batch_size     : nuber of files per batch to process

    """

    exclude_files = set(exclude_files or [])

    datasets = []

    for folder in folders : 

        print(f"Processing folder : {folder}")
        # --- File discovery ---

        all_files = sorted(glob.glob(f'{data_path}/{folder}/OCT_DETECT_AUS_Daily_*.nc'))

        #Apply exclusions

        files = [ f for f in all_files
                  if f not in exclude_files
                  and (not skip_empty_files or os.path.getsize(f) > 0)
                ]
        if not files :
            print(f" No valid files found for folder {folder}, skipping.")
            continue
        
        try:
    
            ds_coarse = xr.open_mfdataset(
                files,
                chunks = chunks,
                parallel = True,
                preprocess = preprocess_fn,
                engine='netcdf' # addition for "runtime error" argo
            ).load()

            datasets.append(ds_coarse)

        except (FileNotFoundError, OSError, ValueError) as e :
            if skip_errors:
    
                # print(f"[SKIPPED] Batch {i//batch_size +1} : {e}")
                # skipped_files.extend(batch)
    
                continue
            else:
                raise
              
        del all_files
        del files
        
    combined = xr.concat(datasets, dim="time")
    # combined = combined.sortby("time")

    return combined

# Regrid OT to match LS grid resolution/coordinates before combining,
# since OT and LS are natively on different grids.
# Method: coarsening and then using 'nearest-neighbour' via [e.g. xesmf / xarray interp]
# OT data is made bool using the threshold of 0.5.

In [ ]:
# For overshooting top regridding and merging

folder_1 = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 20231] #part 1
folder_2 = [20232, 2024] #part 2

pad = 5 # to manage the edge accuracy

LAT_MAX=-10
LAT_MIN=-20
LON_MIN=120
LON_MAX=148

lat_max = -10+pad #start for upto -10
lat_min = -20-pad #stop for upto -20
lon_min = 120-pad #start for upto 120
lon_max = 148+pad #stop for upto 148

# region of interest i.e Australia
# lon_min = 109.895
# lon_max = 158.035
# lat_min = -44.9 #threshold for OT data
# lat_max =  -9.565

ds_1 = xr.open_dataset('/g/data/if69/tp4064/Project_1/OT/data/extracted/2015/OCT_DETECT_AUS_Daily_2015_200.nc', 
                            drop_variables =['ot_probability','grid_lon','grid_lat','time'] )
drop_variables_1 = list(ds_1.variables) # this is indirect way to extract variables for part 1 files


ds_2 = xr.open_dataset('/g/data/if69/tp4064/Project_1/OT/data/extracted/20232/OCT_DETECT_AUS_Daily_2023200.nc', 
                            drop_variables =['maximum_ot_probability','grid_lon','grid_lat','time'] )
drop_variables_2 = list(ds_2.variables) # this is indirect way to extract variables for part2 files



# ds_out = xr.open_dataset('/g/data/if69/tp4064/2015/wztln_201502_UTC.nc')

land = xr.open_dataset("/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/fx/sftlf/latest"+
"/sftlf_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1.nc", drop_variables = ['crs'] ) 
# australian region land fraction file , used from barra because # next I will use parameter from barra
 
ds_out = land  # no rounding for ds_out, it changes the no. of counts while regridding

dataset_1 = process_folders(folders = folder_1,
                           data_path = '/g/data/if69/tp4064/Project_1/OT/data/extracted',
                            preprocess_fn = _preprocess,
                            chunks = {'lat':-1,'lon':-1},
                            skip_errors = False
                           )

dataset_2 = process_folders(
                    folders=folder_2,
                    data_path='/g/data/if69/tp4064/Project_1/OT/data/extracted',
                    preprocess_fn=_preprocess_2,
                    chunks='auto',
                    exclude_files=['/g/data/if69/tp4064/Project_1/OT/data/extracted/2023/OCT_DETECT_AUS_Daily_2023188.nc'],
                    skip_empty_files=True,
                    skip_errors=True,
                )

dataset = xr.concat([dataset_1, dataset_2], dim = 'time').sortby("time")
dataset_gridded = gridding(dataset, ds_out,lat_max,lat_min, lon_min, lon_max)

# All year LS data is merged and converted to boolean using the threshold of 1.

In [ ]:

# Base folder containing yearly folders for lightning data
base_path = "/g/data/if69/tp4064/lightning_data/"

# List of folders (9 years)
folders = [str(y) for y in range(2016, 2025)]  # adjust end year if needed

# Step 2: Collect all files from all folders
lightning_files = []
for folder in folders:
    
    folder_path = os.path.join(base_path, folder)
    print(folder_path)
    files = glob.glob(os.path.join(folder_path, "wztln_*_UTC.nc"))
    lightning_files.extend(files)

# Step 3: Open all files into one Dataset
lightning_data = xr.open_mfdataset(
    lightning_files,
    concat_dim='time',
    combine='nested',
    parallel=True,
    chunks={'time': 24, 'grid_lat': 280, 'grid_lon': 280}  # adjust chunk size as needed
)

lightning_data = (lightning_data['lightning_count'] > 1) # making it bool and with minimum stroke condition of >= 2

# selecting the LS data for given region
# selecting only warm season data for summation at every grid.

In [36]:
warm_ls = lightning_data.sel(lat=slice(LAT_MIN, LAT_MAX),lon=slice(LON_MIN, LON_MAX)).to_dataset()

warm_ls = warm_ls.sel(time=warm_ls['time'].dt.month.isin([10,11,12,1,2,3,4])).sum('time').compute()

warm_ot = dataset_gridded.sel(time=dataset_gridded['time'].dt.month.isin([10,11,12,1,2,3,4])).sum('time').compute()

# saving final data to given folder for figure generations.

In [40]:
warm_ot.to_netcdf("/g/data/if69/tp4064/Project_1/OT/warm_season_NT_data/warm_season_OT.nc")
warm_ds.to_netcdf("/g/data/if69/tp4064/Project_1/Lightning/warm_season_LS_NT/warm_season_LS.nc")